# Module M - Spectral Feature Extraction

Conventional numerical features from AI-estimated reconstructed VIS spectral information.

## Objective and 31-band contract

Module M accepts a cube shaped `(H, W, 31)`. The default bands are 400-700 nm at 10 nm intervals. It does not diagnose disease or measure biological concentrations.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from spectraderm.features.spectral_features import (
    HYPERSKIN_VIS_WAVELENGTHS, extract_spectral_features,
)

## Load reconstructed cube

The local interim cube is preferred. If unavailable, this notebook uses a clearly labelled synthetic cube. Channel-first input is transposed here once, before calling Module M; the extractor itself accepts only HWC.

In [ ]:
cube_path = root / 'data' / 'interim' / 'face_spectral_cube.npy'
if cube_path.exists():
    spectral_cube = np.load(cube_path)
    if spectral_cube.ndim == 3 and spectral_cube.shape[0] == 31 and spectral_cube.shape[-1] != 31:
        spectral_cube = np.transpose(spectral_cube, (1, 2, 0))
    source_label = f'Local reconstructed cube: {cube_path.name}'
else:
    rng = np.random.default_rng(42)
    spectral_cube = rng.random((64, 64, 31), dtype=np.float32)
    source_label = 'Synthetic demonstration cube (no local cube found)'

assert spectral_cube.shape[-1] == 31
print(source_label, spectral_cube.shape)

## Prepare ROI and extract features

This simple center mask is a demonstration ROI only. In a pipeline, an ROI can be supplied by Modules G/H.

In [ ]:
height, width, _ = spectral_cube.shape
mask = np.zeros((height, width), dtype=bool)
mask[height // 4:3 * height // 4, width // 4:3 * width // 4] = True
result = extract_spectral_features(spectral_cube, mask=mask)
print('ROI pixels:', result.roi_pixel_count)
print('Warnings:', result.warnings)

## Feature groups, table, ratios, differences, and slopes

In [ ]:
for group, names in result.feature_groups.items():
    print(f'{group}: {names}')

for name, value in result.features.items():
    print(f'{name:45s} {value:.6g}')

print('\nRatios, differences, and slopes:')
for name, value in result.features.items():
    if name.startswith(('ratio_', 'difference_', 'slope_')):
        print(f'{name:45s} {value:.6g}')

## Spectral signature and investigational proxies

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(result.wavelengths_nm, result.spectral_signature, marker='o')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Regional mean reconstructed value')
plt.title('AI-estimated VIS spectral signature')
plt.grid(True, alpha=0.3)
plt.show()

print('Vascular-related spectral proxy:', result.features['vascular_related_spectral_proxy'])
print('Pigmentation-related spectral proxy:', result.features['pigmentation_related_spectral_proxy'])

## Exclusions, limitations, and safety

The 400-700 nm output is not used for water, hydration, or water-absorption features. A structural/collagen-related proxy is not established and is excluded. The two displayed proxies are investigational spectral appearance proxies, not measurements of hemoglobin, inflammation, melanin, or any biological concentration. Spectral features do not establish disease presence or absence.